In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [3]:
!pip install transformers -q

In [4]:
from transformers import pipeline
print("transformers loaded successfully")

transformers loaded successfully


Sentiment analysis

In [5]:
sentiment = pipeline("sentiment-analysis")

sentences = [
    "Distributed systems are complex but rewarding",
    "This model takes forever to train",
    "I love how transformers handle long range dependencies",
       "This is not good at all",           # Contains 'good' → often POSITIVE
    "I don't hate this",                 # Double negative → often wrong
    "Could be worse",                    # Negative phrasing, mild positive meaning
    "Sick beats bro",                    # 'Sick' is slang for good → often NEGATIVE
    "This model takes forever to train", # Your example
]

for s in sentences:
    result = sentiment(s)
    emoji = "✅" if result[0]['label'] == 'NEGATIVE' else "❌"
    print(f"{emoji} {result[0]['label']} ({result[0]['score']:.2f}) — {s}")

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

❌ POSITIVE (1.00) — Distributed systems are complex but rewarding
❌ POSITIVE (0.99) — This model takes forever to train
❌ POSITIVE (1.00) — I love how transformers handle long range dependencies
✅ NEGATIVE (1.00) — This is not good at all
❌ POSITIVE (1.00) — I don't hate this
✅ NEGATIVE (1.00) — Could be worse
✅ NEGATIVE (0.99) — Sick beats bro
❌ POSITIVE (0.99) — This model takes forever to train


---

## Why This Works Without the Pipeline

###The pipeline is just a wrapper. What it does internally is exactly these 3 steps:

Your text
    ↓
tokenizer()        ← converts text to numbers the model understands
    ↓
model.generate()   ← runs the model, produces output numbers
    ↓
tokenizer.decode() ← converts numbers back to readable text
    ↓
Summary

In [6]:
from transformers import BartForConditionalGeneration, BartTokenizer

model_name = "sshleifer/distilbart-cnn-12-6"
tokenizer = BartTokenizer.from_pretrained(model_name)
model = BartForConditionalGeneration.from_pretrained(model_name)

article = """
Distributed systems are systems in which components located on 
networked computers communicate and coordinate their actions by 
passing messages. The components interact with each other in order 
to achieve a common goal. Three significant challenges of distributed 
systems are maintaining concurrency, overcoming the lack of a global 
clock, and handling component failures independently. Examples of 
distributed systems vary from SOA-based systems to massively 
multiplayer online games to peer-to-peer applications.
"""

inputs = tokenizer(article, return_tensors="pt", max_length=1024, truncation=True)
summary_ids = model.generate(inputs["input_ids"], max_length=60, min_length=20)
summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)

print("Original:", len(article.split()), "words")
print("Summary:", summary)
print("Compressed to:", len(summary.split()), "words")




tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.22G [00:00<?, ?B/s]

Please make sure the generation config includes `forced_bos_token_id=0`. 


model.safetensors:   0%|          | 0.00/1.22G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/358 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Original: 69 words
Summary:  The components interact with each other in order to achieve a common goal . The challenges of distributed systems are maintaining concurrency, overcoming the lack of a global clock, and handling component failures independently .
Compressed to: 35 words


In [7]:
qa = pipeline("question-answering")

context = """
The CAP theorem states that a distributed system can only provide 
two of the following three guarantees simultaneously: Consistency, 
Availability, and Partition tolerance. Consistency means all nodes 
see the same data at the same time. Availability means every request 
receives a response. Partition tolerance means the system continues 
operating despite network failures between nodes.
"""

questions = [
    "What does the CAP theorem state?",
    "What does consistency mean?",
    "What happens during network failures?"
]

for q in questions:
    result = qa(question=q, context=context)
    print(f"Q: {q}")
    print(f"A: {result['answer']} (confidence: {result['score']:.2f})")
    print()

No model was supplied, defaulted to distilbert/distilbert-base-cased-distilled-squad and revision 564e9b5.
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/473 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/261M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Q: What does the CAP theorem state?
A: that a distributed system can only provide 
two of the following three guarantees simultaneously (confidence: 0.43)

Q: What does consistency mean?
A: every request 
receives a response (confidence: 0.67)

Q: What happens during network failures?
A: the system continues 
operating (confidence: 0.45)



In [9]:
classifier = pipeline("zero-shot-classification")

texts = [
    "Kubernetes is crashing our pods in production",
    "The new transformer architecture reduces training time by 40%",
    "Our team is hiring senior engineers for the infra team",
]

# labels = ["incident", "research", "hiring", "product update"]
labels = ["urgent", "not urgent"]

for text in texts:
    result = classifier(text, candidate_labels=labels)
    print(f"Text: {text}")
    print(f"Top label: {result['labels'][0]} ({result['scores'][0]:.2f})")
    print(f"All scores: {dict(zip(result['labels'], [f'{s:.2f}' for s in result['scores']]))}")
    print()

No model was supplied, defaulted to facebook/bart-large-mnli and revision d7645e1.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

Text: Kubernetes is crashing our pods in production
Top label: urgent (0.94)
All scores: {'urgent': '0.94', 'not urgent': '0.06'}

Text: The new transformer architecture reduces training time by 40%
Top label: urgent (0.78)
All scores: {'urgent': '0.78', 'not urgent': '0.22'}

Text: Our team is hiring senior engineers for the infra team
Top label: urgent (0.75)
All scores: {'urgent': '0.75', 'not urgent': '0.25'}



In [10]:
generator = pipeline("text-generation", model="gpt2")

prompts = [
    "The future of distributed systems is",
    "A senior engineer transitioning to AI should",
]

for prompt in prompts:
    result = generator(prompt, max_length=60, num_return_sequences=1)
    print(f"Prompt: {prompt}")
    print(f"Generated: {result[0]['generated_text']}")
    print()

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Passing `generation_config` together with generation-related arguments=({'num_return_sequences', 'max_length'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=60) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=60) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Prompt: The future of distributed systems is
Generated: The future of distributed systems is in flux. While some researchers are already exploring how to implement distributed systems in an elegant way, some find that the concept is a bit too abstract. For example, there is a new approach called "distributed ledger" that seeks to build a distributed ledger, similar to a traditional paper ledger. It is designed to be more flexible and more efficient, but it also incorporates the underlying distributed ledger technology.

The goal of the concept is to achieve a decentralization of the ledger's assets. It is also designed to be easier to implement, and to be more secure than traditional paper. However, some researchers have found that there are some problems with the approach. For example, there is a lack of a way to store the value of a single data point. Moreover, the new technology is not fully proof-of-principle and not fully scalable.

The idea of distributed ledger is to be more tra

# Project 1 — Multi-Task NLP Pipeline

Exploring 5 core NLP tasks using HuggingFace Transformers pipelines.

## Tasks covered
1. Sentiment Analysis — DistilBERT
2. Summarization — BART Large CNN
3. Question Answering — DistilBERT
4. Zero-Shot Classification — BART
5. Text Generation — GPT2

## Key insight
All 5 tasks use the same transformer architecture underneath.
The pipeline API abstracts tokenization, inference, and post-processing.